# DeepRL Monopoly — Builder+DealMaker self-play (no ASU distillation)

Trains a DDQN/PPO agent via pure self-play RL against `TheBuilder` + `TheDealMaker`
(the two strongest fixed-policy opponents). Reward includes a custom
`liquidity_risk` shaping term (see `monopoly_game_engine/env.py`).

ASU is not used anywhere in this notebook — no teacher, no labels, no distillation.

## 1. Mount Drive (for checkpoints) — optional, skip if you don't need persistence

In [5]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Mounted at /content/drive


## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [6]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

/content
Cloning into 'DeepRL_Monopoly'...
remote: Enumerating objects: 747, done.
remote: Counting objects: 100% (747/747), done.
remote: Compressing objects: 100% (288/288), done.
remote: Total 747 (delta 461), reused 733 (delta 447), pack-reused 0 (from 0)
Receiving objects: 100% (747/747), 33.92 MiB | 34.42 MiB/s, done.
Resolving deltas: 100% (461/461), done.
/content/DeepRL_Monopoly


## 3. Check GPU + torch (Colab ships CUDA-enabled torch already)

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU (T4), then re-run this cell.')

## 3.5. Quick timing test (100 games) — run this first

Same measurement we ran locally (CPU-only: ~180s/game once the replay buffer
fills and gradient updates kick in). This tells us the real GPU speedup before
committing to a multi-thousand-game run.

In [7]:
import sys, time
sys.path.insert(0, '.')
from monopoly_game_engine.train import train
from monopoly_game_engine.agent_ddqn import DDQNAgent

agent = DDQNAgent(player_id=0, hybrid=True, device='auto')
print('agent device:', agent.device)

t0 = time.time()
history = train(agent, is_ppo=False, hybrid=True, n_games=100, log_every=10, seed=1)
elapsed = time.time() - t0
print('ELAPSED_SECONDS', elapsed)
print('sec/game', elapsed / 100)
print('local CPU baseline was ~180 sec/game -> speedup:', 180 / (elapsed / 100))

agent device: cuda

Training Hybrid DDQN agent (player 0)
Total games: 100  |  Log every: 10
  Game    10 | Win%:   0.0%  ε=0.995 | Props: 11.1 | Trades init/acc/dec: 117.6/6.0/67.4
  Game    20 | Win%:   0.0%  ε=0.990 | Props: 12.8 | Trades init/acc/dec: 94.0/4.5/38.5
  Game    30 | Win%:   0.0%  ε=0.985 | Props: 11.4 | Trades init/acc/dec: 81.2/4.3/37.4
  Game    40 | Win%:   0.0%  ε=0.980 | Props: 11.0 | Trades init/acc/dec: 50.8/6.1/37.3
  Game    50 | Win%:   0.0%  ε=0.975 | Props: 13.8 | Trades init/acc/dec: 89.9/11.3/48.3
  Game    60 | Win%:   0.0%  ε=0.970 | Props: 11.3 | Trades init/acc/dec: 55.2/7.8/39.6
  Game    70 | Win%:   0.0%  ε=0.966 | Props: 10.0 | Trades init/acc/dec: 64.4/5.3/46.2
  Game    80 | Win%:   0.0%  ε=0.961 | Props: 12.3 | Trades init/acc/dec: 72.0/3.8/42.2
  Game    90 | Win%:   0.0%  ε=0.956 | Props: 13.0 | Trades init/acc/dec: 84.5/7.7/27.7
  Game   100 | Win%:   0.0%  ε=0.951 | Props: 9.6 | Trades init/acc/dec: 97.7/5.1/27.7
ELAPSED_SECONDS 690.135211

## 4. Train

`--device auto` picks CUDA automatically. `--checkpoint-every` saves a resumable
checkpoint (format 3: optimizer state + replay buffer + progress) so a disconnected
Colab session can `--resume` from the same `--out` path.

In [8]:
import os
OUT = f"{CHECKPOINT_DIR}/ddqn_builder_dealmaker.pt" if 'CHECKPOINT_DIR' in dir() else 'artifacts/ddqn_builder_dealmaker.pt'
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 2000 --device auto \
  --checkpoint-every 100 \
  --out "{OUT}"


  Algorithm : DDQN
  Mode      : Hybrid
  Games     : 5000
  Device    : auto
  Save to   : /content/drive/MyDrive/DeepRL_Monopoly_ckpt/ddqn_builder_dealmaker.pt


Training Hybrid DDQN agent (player 0)
Total games: 5000  |  Log every: 100
Traceback (most recent call last):
  File "/content/DeepRL_Monopoly/tools/train_and_save.py", line 228, in <module>
  File "/content/DeepRL_Monopoly/tools/train_and_save.py", line 176, in main
    agent, history = train_ddqn(
                     ^^^^^^^^^^^
  File "/content/DeepRL_Monopoly/monopoly_game_engine/__init__.py", line 85, in train_ddqn
    history = train(
              ^^^^^^
  File "/content/DeepRL_Monopoly/monopoly_game_engine/train.py", line 360, in train
    result = run_episode(env, learning_agent, fp_agents, agent_pid, is_ppo)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/DeepRL_Monopoly/monopoly_game_engine/train.py", line 153, in run_episode
    update_stats = learning_agent.update()

## 5. Resume after a disconnect (same --out path, --resume flag)

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 20000 --device auto --seed 42 --resume \
  --checkpoint-every 100 \
  --out "{OUT}"

## 6. Quick eval against Builder + DealMaker after training

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/play_game.py \
  --algo ddqn --players 4 --model "{OUT}"